## Pursuit / Evasion Scenario

Two UAVs:
- **Evader** (sysid=1, GREEN): Attempts to hover at a fixed point. ArduPilot's built-in ADSB obstacle avoidance causes it to drift away when the pursuer gets close.
- **Pursuer** (sysid=255, RED): Arms, takes off, then continuously flies toward the evader's last known position as broadcast via Remote ID.

In [ ]:
from simulator import Simulator
from simulator.config import PARAMS_PATH, Color
from simulator.entities import SimVehicle
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import GuidedPlan
from simulator.planner.plans.pursuit import PursuitPlan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
)

clean()

## Simulation Positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

evader_home = ENUPose(15, -10, 0, 0)
pursuer_home = ENUPose(15, 10, 0, 0)  # 20m north of evader

hover_wp = ENU(0, 0, 5)  # hover point relative to evader home
speed = 3.0  # m/s, same for both vehicles

## Create Vehicles

In [ ]:
# Evader (sysid=1, GREEN)
# Tries to hold position; ArduPilot ADSB avoidance will push it away when threatened.
evader_plan = GuidedPlan.from_relative_path(
    name="hover_plan",
    relative_path=[hover_wp],
    enu_origin=enu_origin,
    relative_home=evader_home,
    land=False,
    navigation_speed=speed,
)
evader = SimVehicle.from_relative(
    sysid=1,
    gcs_name=f"GREEN_{Color.GREEN.emoji}",
    plan=evader_plan,
    color=Color.GREEN,
    enu_origin=enu_origin,
    relative_home=evader_home,
    relative_path=[hover_wp],
    model="gazebo-iris",
    firmware="ArduCopter",
)

# Pursuer (sysid=255, RED)
# Continuously flies toward the evader's Remote ID position.
pursuer_plan = PursuitPlan(
    name="pursuit_plan",
    target_sysid=1,
    speed=speed,
    takeoff_alt=5.0,
)
pursuer = SimVehicle.from_relative(
    sysid=255,
    gcs_name="RED_\U0001f7e5",
    plan=pursuer_plan,
    color=Color.RED,
    enu_origin=enu_origin,
    relative_home=pursuer_home,
    relative_path=[],  # dynamic — no predefined waypoints
    model="gazebo-iris",
    # model="gazebo-zephyr",
    # firmware="ArduPlane",
)

## Visualizer

### Gazebo

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_gaz = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_gaz)

### QGroundControl

In [ ]:
qgc = QGC(gra_origin)
origin_qgc = QGCMarker(
    name="origin",
    pos=gra_origin.unpose(),
    color=Color.WHITE,
)
qgc.markers.append(origin_qgc)

### No Visualizer

In [ ]:
novis = NoVisualizer(gra_origin)

## Simulator

In [ ]:
simulator = Simulator(
    visualizer=gaz,
    terminals=[SimProcess.GCS],
    verbose=1,
)

parm_paths = {
    1: str(PARAMS_PATH / "vehicle.parm"),
    255: str(PARAMS_PATH / "mallicious.parm"),
}
simulator.add_vehicle(evader, parm=parm_paths[1])
simulator.add_vehicle(pursuer, parm=parm_paths[255])

simulator.show()

## Run

In [ ]:
orac = simulator.launch()
orac.run()